In [ ]:
# THIS IS MAIN CODE WITH EXCEL FORMAT CHANGED - Latest from claude Signalization plan added

import os
import re
import sys
import traceback
from collections import defaultdict

import pdfplumber
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# ------------------ PATHS (YOUR INPUTS) ------------------
PDF_PATH = r"C:/Abid/Abid/AI AGENTS/qc PDFS/438262.pdf"
OUT_XLSX = r"C:/Abid/Abid/AI AGENTS/qc PDFS/438262_1.xlsx"
OUT_CSV_FALLBACK = r"C:/Abid/Abid/AI AGENTS/qc PDFS/pay_item_quantities_fallback.csv"

# ------------------ PATTERNS ------------------
PAY_ITEM_RE = re.compile(r"^\d{3}-\d{1,3}-\d{1,3}$")  # catches 700-141-140
NUM_TOKEN_RE = re.compile(r"^(?:\d{1,4}|\d,\d{3})(?:\.\d+)?$")

UNITS = {"LF", "EA", "MI", "SF", "AS", "LS", "LU", "DA", "PI", "HR", "LO"}
RUNS_WORDS = {"RUNS", "RUN"}
AT_WORDS = {"@", "AT"}

# For reading sheet numbers at bottom-right (T-4, IT-2, S-10, etc.)
SHEET_ID_RE = re.compile(r"\b[A-Z]{1,3}\s*-\s*\d+\b", re.IGNORECASE)

# ------------------ TUNABLE PARAMETERS ------------------
Y_TOL = 4.0
MAX_HORIZONTAL_DISTANCE = 250
UNIT_GAP_MAX = 70
VERT_GAP_MAX = 20   # max vertical gap (pts) between a stacked qty and its pay item
H_ALIGN_TOL = 30    # how far (pts) the qty can sit past the pay item's left/right edge


def autofit_columns(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


def normalize_sheet_id(text):
    return re.sub(r"\s+", "", text.upper())


def extract_sheet_ids(pdf_path):
    """
    Reads bottom-right title block area of each page and extracts sheet IDs like IT-2.
    Returns: dict {page_number: sheet_id}
    """
    page_to_sheet = {}

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            w, h = page.width, page.height

            # Bottom-right crop region
            crop_box = (
                w * 0.78,   # x0
                h * 0.82,   # top
                w * 0.995,  # x1
                h * 0.995   # bottom
            )

            try:
                cropped = page.crop(crop_box)
                text = cropped.extract_text() or ""
            except Exception:
                text = ""

            m = SHEET_ID_RE.search(text)
            if m:
                page_to_sheet[page_num] = normalize_sheet_id(m.group(0))
            else:
                page_to_sheet[page_num] = f"PAGE-{page_num}"

    return page_to_sheet


def find_runs_patterns(words):
    """
    Scan words for the pattern: NUMBER RUNS @ NUMBER UNIT
    e.g. "2 RUNS @ 129 LF"

    Returns:
        runs_pairs   : list of (effective_qty, unit_word, anchor_word)
                       where anchor_word is the run-count word (used for
                       proximity matching to the pay item)
        excluded_idx : set of word indices that are consumed by a RUNS pattern
                       (so they are NOT matched individually as plain numbers)
    """
    runs_pairs = []
    excluded_idx = set()

    for i in range(len(words) - 4):
        w0 = words[i]       # number  (run count)
        w1 = words[i + 1]   # RUNS / RUN
        w2 = words[i + 2]   # @  / AT
        w3 = words[i + 3]   # number  (unit quantity)
        w4 = words[i + 4]   # unit

        if (NUM_TOKEN_RE.match(w0["text"]) and
                w1["text"].upper() in RUNS_WORDS and
                w2["text"].upper() in AT_WORDS and
                NUM_TOKEN_RE.match(w3["text"]) and
                w4["text"].upper() in UNITS):

            run_count = float(w0["text"].replace(",", ""))
            unit_qty  = float(w3["text"].replace(",", ""))
            effective_qty = run_count * unit_qty

            # Use the run-count word as the anchor so that when the pay item
            # is to the LEFT of "2 RUNS @ 57 LF", the "2" position is used
            # for proximity — giving the correct closest match.
            # Quantity is just the unit qty (e.g. 129 LF), NOT multiplied by run count.
            # The run count (w0) is excluded so it isn't mistakenly matched as the qty.
            runs_pairs.append((unit_qty, w4, w0))
            excluded_idx.add(i)   # suppress only the run count from plain matching

    return runs_pairs, excluded_idx


def pair_bbox(pair):
    """Bounding box, quantity and unit for a plain or RUNS pair."""
    if pair[0] == "__runs__":
        _, unit_w, qty, first = pair
    else:
        first, unit_w = pair
        qty = float(first["text"].replace(",", ""))
    return {
        "x0": first["x0"],
        "x1": unit_w["x1"],
        "top": min(first["top"], unit_w["top"]),
        "bottom": max(first["bottom"], unit_w["bottom"]),
        "anchor": first,
        "qty": qty,
        "unit": unit_w["text"].upper(),
    }


def extract_occurrences(pdf_path):
    """
    Extract occurrences across ALL pages.
    Each occurrence is a row: {page, pay_item, qty, unit}
    """
    occurrences = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"[INFO] Opened PDF. Pages: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num % 5 == 1 or page_num == total_pages:
                print(f"[INFO] Processing page {page_num}/{total_pages} ...")

            words = page.extract_words(keep_blank_chars=False, use_text_flow=True) or []
            if not words:
                continue

            pay_items, numbers, units = [], [], []

            for idx, w in enumerate(words):
                text = (w.get("text") or "").strip()
                if not text:
                    continue
                if PAY_ITEM_RE.match(text):
                    pay_items.append(w)
                elif text.upper() in UNITS:
                    units.append(w)
                elif NUM_TOKEN_RE.match(text):
                    numbers.append((idx, w))

            # --- Detect "N RUNS @ QTY UNIT" patterns first ---
            runs_pairs, excluded_idx = find_runs_patterns(words)

            # Build valid (number + unit) pairs from plain numbers,
            # skipping any number that belongs to a RUNS pattern.
            pairs = []
            for idx, q in numbers:
                if idx in excluded_idx:
                    continue
                qy = q["top"]
                for u in units:
                    if abs(u["top"] - qy) <= Y_TOL:
                        gap = u["x0"] - q["x1"]
                        if 0 <= gap <= UNIT_GAP_MAX:
                            pairs.append((q, u))

            # Add RUNS-derived pairs: (anchor_word, unit_word) with effective qty stored
            # We store them as dicts with synthetic qty baked in so the candidate
            # loop below can treat them uniformly.
            for eff_qty, unit_w, anchor_w in runs_pairs:
                pairs.append(("__runs__", unit_w, eff_qty, anchor_w))

            # Assign quantities to pay items
            boxes = [pair_bbox(p) for p in pairs]
            results = {}   # pay item index -> pair index
            used = set()   # pairs already claimed

            # Pass 1: quantity on the same line (original behavior)
            for i, pi in enumerate(pay_items):
                pi_center = (pi["x0"] + pi["x1"]) / 2
                best = None
                for j, b in enumerate(boxes):
                    a = b["anchor"]
                    if abs(a["top"] - pi["top"]) > Y_TOL:
                        continue
                    dx = abs((a["x0"] + a["x1"]) / 2 - pi_center)
                    if dx <= MAX_HORIZONTAL_DISTANCE and (best is None or dx < best[0]):
                        best = (dx, j)
                if best:
                    results[i] = best[1]
                    used.add(best[1])

            # Pass 2: quantity stacked above (preferred) or below the pay item
            for i, pi in enumerate(pay_items):
                if i in results:
                    continue
                best = None
                for j, b in enumerate(boxes):
                    if j in used:
                        continue
                    b_center = (b["x0"] + b["x1"]) / 2
                    if not (pi["x0"] - H_ALIGN_TOL <= b_center <= pi["x1"] + H_ALIGN_TOL):
                        continue
                    gap_above = pi["top"] - b["bottom"]
                    gap_below = b["top"] - pi["bottom"]
                    if 0 <= gap_above <= VERT_GAP_MAX:
                        score = gap_above
                    elif 0 <= gap_below <= VERT_GAP_MAX:
                        score = gap_below + VERT_GAP_MAX
                    else:
                        continue
                    if best is None or score < best[0]:
                        best = (score, j)
                if best:
                    results[i] = best[1]
                    used.add(best[1])

            for i, pi in enumerate(pay_items):
                j = results.get(i)
                occurrences.append({
                    "page": page_num,
                    "pay_item": pi["text"],
                    "qty": boxes[j]["qty"] if j is not None else None,
                    "unit": boxes[j]["unit"] if j is not None else None,
                })

    print(f"[INFO] Extracted occurrences: {len(occurrences)}")
    return occurrences


def compute_totals(occurrences):
    totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        totals[(o["pay_item"], o["unit"])] += o["qty"]
    return totals


def organize_occurrences_expressions(occurrences):
    overall_list = defaultdict(list)
    per_page_list = defaultdict(list)

    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        overall_list[(o["pay_item"], o["unit"])].append(o["qty"])
        per_page_list[(o["page"], o["pay_item"], o["unit"])].append(o["qty"])

    def fmt(v):
        return str(int(v)) if float(v).is_integer() else str(v)

    overall_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in overall_list.items()}
    per_page_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in per_page_list.items()}

    return overall_expr, per_page_expr


def sort_sheet_ids(sheet_ids):
    def key_func(s):
        m = re.match(r"([A-Z]+)-(\d+)", s)
        if m:
            return (m.group(1), int(m.group(2)))
        return (s, 999999)
    return sorted(sheet_ids, key=key_func)


def build_per_sheet_expr(per_page_expr, page_to_sheet):
    """
    Convert per-page expressions to per-sheet expressions using sheet IDs.
    Output: {(sheet_id, pay_item, unit): "36+13+273"}
    """
    grouped = defaultdict(list)

    for (page, pay_item, unit), expr in per_page_expr.items():
        sheet_id = page_to_sheet.get(page, f"PAGE-{page}")
        grouped[(sheet_id, pay_item, unit)].append(expr)

    merged = {}
    for key, exprs in grouped.items():
        parts = []
        for e in exprs:
            if e:
                parts.extend(e.split("+"))
        merged[key] = "+".join(parts)

    return merged


def apply_table_border(ws, row_start, row_end, col_start, col_end):
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for r in range(row_start, row_end + 1):
        for c in range(col_start, col_end + 1):
            ws.cell(r, c).border = border


def write_excel_tabulation(occurrences, totals, overall_expr, per_sheet_expr, page_to_sheet, out_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Tabulation"

    # ---------------- Styles ----------------
    header_font = Font(name="Arial", size=11, bold=True, italic=True)
    body_font = Font(name="Arial", size=10, italic=True)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center", wrap_text=True)

    # ---------------- Data prep ----------------
    sheet_ids = sort_sheet_ids(sorted(set(page_to_sheet.values())))
    pay_item_keys = sorted(set((o["pay_item"], o["unit"]) for o in occurrences if o["qty"] is not None and o["unit"] is not None))

    # ---------------- Layout ----------------
    start_row = 1

    ws.merge_cells(start_row=1, start_column=4, end_row=1, end_column=3 + len(sheet_ids) * 2)
    ws.cell(1, 4).value = "TABULATION OF QUANTITIES"
    ws.cell(1, 4).font = header_font
    ws.cell(1, 4).alignment = center

    ws.merge_cells(start_row=2, start_column=4, end_row=2, end_column=3 + len(sheet_ids) * 2)
    ws.cell(2, 4).value = "SHEET NUMBERS"
    ws.cell(2, 4).font = header_font
    ws.cell(2, 4).alignment = center

    ws.merge_cells("A3:A4")
    ws["A3"] = "PAY\nITEM\nNO."
    ws["A3"].font = header_font
    ws["A3"].alignment = center

    ws.merge_cells("B3:B4")
    ws["B3"] = "DESCRIPTION"
    ws["B3"].font = header_font
    ws["B3"].alignment = center

    ws.merge_cells("C3:C4")
    ws["C3"] = "UNIT"
    ws["C3"].font = header_font
    ws["C3"].alignment = center

    col = 4
    sheet_plan_col = {}
    for sid in sheet_ids:
        ws.merge_cells(start_row=3, start_column=col, end_row=3, end_column=col + 1)
        ws.cell(3, col).value = sid
        ws.cell(3, col).font = header_font
        ws.cell(3, col).alignment = center

        ws.cell(4, col).value = "PLAN"
        ws.cell(4, col + 1).value = "FINAL"
        ws.cell(4, col).font = header_font
        ws.cell(4, col + 1).font = header_font
        ws.cell(4, col).alignment = center
        ws.cell(4, col + 1).alignment = center

        sheet_plan_col[sid] = col
        col += 2

    grand_plan_col = col
    ws.merge_cells(start_row=3, start_column=grand_plan_col, end_row=3, end_column=grand_plan_col + 1)
    ws.cell(3, grand_plan_col).value = "GRAND\nTOTAL"
    ws.cell(3, grand_plan_col).font = header_font
    ws.cell(3, grand_plan_col).alignment = center

    ws.cell(4, grand_plan_col).value = "PLAN"
    ws.cell(4, grand_plan_col + 1).value = "FINAL"
    ws.cell(4, grand_plan_col).font = header_font
    ws.cell(4, grand_plan_col + 1).font = header_font
    ws.cell(4, grand_plan_col).alignment = center
    ws.cell(4, grand_plan_col + 1).alignment = center

    # ---------------- Fill rows ----------------
    row = 5
    for pay_item, unit in pay_item_keys:
        ws.cell(row, 1).value = pay_item
        ws.cell(row, 2).value = ""
        ws.cell(row, 3).value = unit

        ws.cell(row, 1).font = body_font
        ws.cell(row, 2).font = body_font
        ws.cell(row, 3).font = body_font
        ws.cell(row, 1).alignment = center
        ws.cell(row, 2).alignment = left_align
        ws.cell(row, 3).alignment = center

        for sid in sheet_ids:
            expr = per_sheet_expr.get((sid, pay_item, unit))
            if expr:
                ws.cell(row, sheet_plan_col[sid]).value = f"={expr}"

        overall = overall_expr.get((pay_item, unit))
        if overall:
            ws.cell(row, grand_plan_col).value = f"={overall}"

        row += 1

    # ---------------- Formatting ----------------
    for r in range(5, row):
        for c in range(1, grand_plan_col + 2):
            ws.cell(r, c).font = body_font
            if c == 2:
                ws.cell(r, c).alignment = left_align
            else:
                ws.cell(r, c).alignment = center

    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 55
    ws.column_dimensions["C"].width = 8

    for c in range(4, grand_plan_col + 2):
        ws.column_dimensions[get_column_letter(c)].width = 10

    apply_table_border(ws, 3, row - 1, 1, grand_plan_col + 1)

    ws2 = wb.create_sheet("Occurrences")
    ws2.append(["page", "sheet_id", "pay_item", "qty", "unit"])
    for o in occurrences:
        ws2.append([o["page"], page_to_sheet.get(o["page"], f"PAGE-{o['page']}"), o["pay_item"], o["qty"], o["unit"]])
    autofit_columns(ws2)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wb.save(out_path)


def write_csv_fallback(occurrences, path):
    import csv
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["page", "pay_item", "qty", "unit"])
        for o in occurrences:
            w.writerow([o["page"], o["pay_item"], o["qty"], o["unit"]])


def main():
    print("[INFO] Starting QC quantity extraction...")

    if not os.path.exists(PDF_PATH):
        print(f"[ERROR] PDF not found:\n  {PDF_PATH}")
        sys.exit(1)

    occurrences = extract_occurrences(PDF_PATH)
    totals = compute_totals(occurrences)
    overall_expr, per_page_expr = organize_occurrences_expressions(occurrences)

    page_to_sheet = extract_sheet_ids(PDF_PATH)
    per_sheet_expr = build_per_sheet_expr(per_page_expr, page_to_sheet)

    try:
        write_excel_tabulation(
            occurrences=occurrences,
            totals=totals,
            overall_expr=overall_expr,
            per_sheet_expr=per_sheet_expr,
            page_to_sheet=page_to_sheet,
            out_path=OUT_XLSX
        )
        print(f"[OK] Excel written:\n  {OUT_XLSX}")
    except Exception as e:
        print("[ERROR] Failed to write Excel. Writing CSV fallback instead.")
        print("Reason:", repr(e))
        write_csv_fallback(occurrences, OUT_CSV_FALLBACK)
        print(f"[OK] CSV fallback written:\n  {OUT_CSV_FALLBACK}")
        print("\nFull traceback:")
        traceback.print_exc()

    print("[INFO] Done.")


if __name__ == "__main__":
    main()

[INFO] Starting QC quantity extraction...
[INFO] Opened PDF. Pages: 16
[INFO] Processing page 1/16 ...


In [2]:
# THIS IS MAIN CODE WITH EXCEL FORMAT CHANGED - Latest from claude

import os
import re
import sys
import traceback
from collections import defaultdict

import pdfplumber
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# ------------------ PATHS (YOUR INPUTS) ------------------
PDF_PATH = r"C:/Abid/Abid/AI AGENTS/qc PDFS/44033825201.pdf"
OUT_XLSX = r"C:/Abid/Abid/AI AGENTS/qc PDFS/44038.xlsx"
OUT_CSV_FALLBACK = r"C:/Abid/Abid/AI AGENTS/qc PDFS/pay_item_quantities_fallback.csv"

# ------------------ PATTERNS ------------------
PAY_ITEM_RE = re.compile(r"^\d{3}-\d{1,3}-\d{1,3}$")  # catches 700-141-140
NUM_TOKEN_RE = re.compile(r"^(?:\d{1,4}|\d,\d{3})(?:\.\d+)?$")

UNITS = {"LF", "EA", "MI", "SF", "AS", "LS", "LU", "DA", "PI", "HR", "LO"}
RUNS_WORDS = {"RUNS", "RUN"}
AT_WORDS = {"@", "AT"}

# For reading sheet numbers at bottom-right
SHEET_ID_RE = re.compile(r"\bIT\s*-\s*\d+\b", re.IGNORECASE)

# ------------------ TUNABLE PARAMETERS ------------------
Y_TOL = 4.0
MAX_HORIZONTAL_DISTANCE = 250
UNIT_GAP_MAX = 70


def autofit_columns(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


def normalize_sheet_id(text):
    return re.sub(r"\s+", "", text.upper())


def extract_sheet_ids(pdf_path):
    """
    Reads bottom-right title block area of each page and extracts sheet IDs like IT-2.
    Returns: dict {page_number: sheet_id}
    """
    page_to_sheet = {}

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            w, h = page.width, page.height

            # Bottom-right crop region
            crop_box = (
                w * 0.78,   # x0
                h * 0.82,   # top
                w * 0.995,  # x1
                h * 0.995   # bottom
            )

            try:
                cropped = page.crop(crop_box)
                text = cropped.extract_text() or ""
            except Exception:
                text = ""

            m = SHEET_ID_RE.search(text)
            if m:
                page_to_sheet[page_num] = normalize_sheet_id(m.group(0))
            else:
                page_to_sheet[page_num] = f"PAGE-{page_num}"

    return page_to_sheet


def find_runs_patterns(words):
    """
    Scan words for the pattern: NUMBER RUNS @ NUMBER UNIT
    e.g. "2 RUNS @ 129 LF"

    Returns:
        runs_pairs   : list of (effective_qty, unit_word, anchor_word)
                       where anchor_word is the run-count word (used for
                       proximity matching to the pay item)
        excluded_idx : set of word indices that are consumed by a RUNS pattern
                       (so they are NOT matched individually as plain numbers)
    """
    runs_pairs = []
    excluded_idx = set()

    for i in range(len(words) - 4):
        w0 = words[i]       # number  (run count)
        w1 = words[i + 1]   # RUNS / RUN
        w2 = words[i + 2]   # @  / AT
        w3 = words[i + 3]   # number  (unit quantity)
        w4 = words[i + 4]   # unit

        if (NUM_TOKEN_RE.match(w0["text"]) and
                w1["text"].upper() in RUNS_WORDS and
                w2["text"].upper() in AT_WORDS and
                NUM_TOKEN_RE.match(w3["text"]) and
                w4["text"].upper() in UNITS):

            run_count = float(w0["text"].replace(",", ""))
            unit_qty  = float(w3["text"].replace(",", ""))
            effective_qty = run_count * unit_qty

            # Use the run-count word as the anchor so that when the pay item
            # is to the LEFT of "2 RUNS @ 57 LF", the "2" position is used
            # for proximity — giving the correct closest match.
            # Quantity is just the unit qty (e.g. 129 LF), NOT multiplied by run count.
            # The run count (w0) is excluded so it isn't mistakenly matched as the qty.
            runs_pairs.append((unit_qty, w4, w0))
            excluded_idx.add(i)   # suppress only the run count from plain matching

    return runs_pairs, excluded_idx


def extract_occurrences(pdf_path):
    """
    Extract occurrences across ALL pages.
    Each occurrence is a row: {page, pay_item, qty, unit}
    """
    occurrences = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"[INFO] Opened PDF. Pages: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num % 5 == 1 or page_num == total_pages:
                print(f"[INFO] Processing page {page_num}/{total_pages} ...")

            words = page.extract_words(keep_blank_chars=False, use_text_flow=True) or []
            if not words:
                continue

            pay_items, numbers, units = [], [], []

            for idx, w in enumerate(words):
                text = (w.get("text") or "").strip()
                if not text:
                    continue
                if PAY_ITEM_RE.match(text):
                    pay_items.append(w)
                elif text.upper() in UNITS:
                    units.append(w)
                elif NUM_TOKEN_RE.match(text):
                    numbers.append((idx, w))

            # --- Detect "N RUNS @ QTY UNIT" patterns first ---
            runs_pairs, excluded_idx = find_runs_patterns(words)

            # Build valid (number + unit) pairs from plain numbers,
            # skipping any number that belongs to a RUNS pattern.
            pairs = []
            for idx, q in numbers:
                if idx in excluded_idx:
                    continue
                qy = q["top"]
                for u in units:
                    if abs(u["top"] - qy) <= Y_TOL:
                        gap = u["x0"] - q["x1"]
                        if 0 <= gap <= UNIT_GAP_MAX:
                            pairs.append((q, u))

            # Add RUNS-derived pairs: (anchor_word, unit_word) with effective qty stored
            # We store them as dicts with synthetic qty baked in so the candidate
            # loop below can treat them uniformly.
            for eff_qty, unit_w, anchor_w in runs_pairs:
                pairs.append(("__runs__", unit_w, eff_qty, anchor_w))

            # Assign closest pair to each pay item
            for pi in pay_items:
                piy = pi["top"]
                pi_center = (pi["x0"] + pi["x1"]) / 2

                candidates = []
                for pair in pairs:
                    if pair[0] == "__runs__":
                        _, u, eff_qty, anchor_w = pair
                        anchor_y = anchor_w["top"]
                        if abs(anchor_y - piy) <= Y_TOL:
                            anchor_center = (anchor_w["x0"] + anchor_w["x1"]) / 2
                            dx = abs(anchor_center - pi_center)
                            if dx <= MAX_HORIZONTAL_DISTANCE:
                                candidates.append((dx, eff_qty, u["text"].upper()))
                    else:
                        q, u = pair
                        if abs(q["top"] - piy) <= Y_TOL:
                            q_center = (q["x0"] + q["x1"]) / 2
                            dx = abs(q_center - pi_center)
                            if dx <= MAX_HORIZONTAL_DISTANCE:
                                candidates.append((dx, float(q["text"].replace(",", "")), u["text"].upper()))

                if not candidates:
                    occurrences.append({
                        "page": page_num,
                        "pay_item": pi["text"],
                        "qty": None,
                        "unit": None
                    })
                    continue

                candidates.sort(key=lambda x: x[0])
                _, qty, unit = candidates[0]

                occurrences.append({
                    "page": page_num,
                    "pay_item": pi["text"],
                    "qty": qty,
                    "unit": unit
                })

    print(f"[INFO] Extracted occurrences: {len(occurrences)}")
    return occurrences


def compute_totals(occurrences):
    totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        totals[(o["pay_item"], o["unit"])] += o["qty"]
    return totals


def organize_occurrences_expressions(occurrences):
    overall_list = defaultdict(list)
    per_page_list = defaultdict(list)

    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        overall_list[(o["pay_item"], o["unit"])].append(o["qty"])
        per_page_list[(o["page"], o["pay_item"], o["unit"])].append(o["qty"])

    def fmt(v):
        return str(int(v)) if float(v).is_integer() else str(v)

    overall_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in overall_list.items()}
    per_page_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in per_page_list.items()}

    return overall_expr, per_page_expr


def sort_sheet_ids(sheet_ids):
    def key_func(s):
        m = re.match(r"([A-Z]+)-(\d+)", s)
        if m:
            return (m.group(1), int(m.group(2)))
        return (s, 999999)
    return sorted(sheet_ids, key=key_func)


def build_per_sheet_expr(per_page_expr, page_to_sheet):
    """
    Convert per-page expressions to per-sheet expressions using sheet IDs.
    Output: {(sheet_id, pay_item, unit): "36+13+273"}
    """
    grouped = defaultdict(list)

    for (page, pay_item, unit), expr in per_page_expr.items():
        sheet_id = page_to_sheet.get(page, f"PAGE-{page}")
        grouped[(sheet_id, pay_item, unit)].append(expr)

    merged = {}
    for key, exprs in grouped.items():
        parts = []
        for e in exprs:
            if e:
                parts.extend(e.split("+"))
        merged[key] = "+".join(parts)

    return merged


def apply_table_border(ws, row_start, row_end, col_start, col_end):
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for r in range(row_start, row_end + 1):
        for c in range(col_start, col_end + 1):
            ws.cell(r, c).border = border


def write_excel_tabulation(occurrences, totals, overall_expr, per_sheet_expr, page_to_sheet, out_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Tabulation"

    # ---------------- Styles ----------------
    header_font = Font(name="Arial", size=11, bold=True, italic=True)
    body_font = Font(name="Arial", size=10, italic=True)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center", wrap_text=True)

    # ---------------- Data prep ----------------
    sheet_ids = sort_sheet_ids(sorted(set(page_to_sheet.values())))
    pay_item_keys = sorted(set((o["pay_item"], o["unit"]) for o in occurrences if o["qty"] is not None and o["unit"] is not None))

    # ---------------- Layout ----------------
    start_row = 1

    ws.merge_cells(start_row=1, start_column=4, end_row=1, end_column=3 + len(sheet_ids) * 2)
    ws.cell(1, 4).value = "TABULATION OF QUANTITIES"
    ws.cell(1, 4).font = header_font
    ws.cell(1, 4).alignment = center

    ws.merge_cells(start_row=2, start_column=4, end_row=2, end_column=3 + len(sheet_ids) * 2)
    ws.cell(2, 4).value = "SHEET NUMBERS"
    ws.cell(2, 4).font = header_font
    ws.cell(2, 4).alignment = center

    ws.merge_cells("A3:A4")
    ws["A3"] = "PAY\nITEM\nNO."
    ws["A3"].font = header_font
    ws["A3"].alignment = center

    ws.merge_cells("B3:B4")
    ws["B3"] = "DESCRIPTION"
    ws["B3"].font = header_font
    ws["B3"].alignment = center

    ws.merge_cells("C3:C4")
    ws["C3"] = "UNIT"
    ws["C3"].font = header_font
    ws["C3"].alignment = center

    col = 4
    sheet_plan_col = {}
    for sid in sheet_ids:
        ws.merge_cells(start_row=3, start_column=col, end_row=3, end_column=col + 1)
        ws.cell(3, col).value = sid
        ws.cell(3, col).font = header_font
        ws.cell(3, col).alignment = center

        ws.cell(4, col).value = "PLAN"
        ws.cell(4, col + 1).value = "FINAL"
        ws.cell(4, col).font = header_font
        ws.cell(4, col + 1).font = header_font
        ws.cell(4, col).alignment = center
        ws.cell(4, col + 1).alignment = center

        sheet_plan_col[sid] = col
        col += 2

    grand_plan_col = col
    ws.merge_cells(start_row=3, start_column=grand_plan_col, end_row=3, end_column=grand_plan_col + 1)
    ws.cell(3, grand_plan_col).value = "GRAND\nTOTAL"
    ws.cell(3, grand_plan_col).font = header_font
    ws.cell(3, grand_plan_col).alignment = center

    ws.cell(4, grand_plan_col).value = "PLAN"
    ws.cell(4, grand_plan_col + 1).value = "FINAL"
    ws.cell(4, grand_plan_col).font = header_font
    ws.cell(4, grand_plan_col + 1).font = header_font
    ws.cell(4, grand_plan_col).alignment = center
    ws.cell(4, grand_plan_col + 1).alignment = center

    # ---------------- Fill rows ----------------
    row = 5
    for pay_item, unit in pay_item_keys:
        ws.cell(row, 1).value = pay_item
        ws.cell(row, 2).value = ""
        ws.cell(row, 3).value = unit

        ws.cell(row, 1).font = body_font
        ws.cell(row, 2).font = body_font
        ws.cell(row, 3).font = body_font
        ws.cell(row, 1).alignment = center
        ws.cell(row, 2).alignment = left_align
        ws.cell(row, 3).alignment = center

        for sid in sheet_ids:
            expr = per_sheet_expr.get((sid, pay_item, unit))
            if expr:
                ws.cell(row, sheet_plan_col[sid]).value = f"={expr}"

        overall = overall_expr.get((pay_item, unit))
        if overall:
            ws.cell(row, grand_plan_col).value = f"={overall}"

        row += 1

    # ---------------- Formatting ----------------
    for r in range(5, row):
        for c in range(1, grand_plan_col + 2):
            ws.cell(r, c).font = body_font
            if c == 2:
                ws.cell(r, c).alignment = left_align
            else:
                ws.cell(r, c).alignment = center

    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 55
    ws.column_dimensions["C"].width = 8

    for c in range(4, grand_plan_col + 2):
        ws.column_dimensions[get_column_letter(c)].width = 10

    apply_table_border(ws, 3, row - 1, 1, grand_plan_col + 1)

    ws2 = wb.create_sheet("Occurrences")
    ws2.append(["page", "sheet_id", "pay_item", "qty", "unit"])
    for o in occurrences:
        ws2.append([o["page"], page_to_sheet.get(o["page"], f"PAGE-{o['page']}"), o["pay_item"], o["qty"], o["unit"]])
    autofit_columns(ws2)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wb.save(out_path)


def write_csv_fallback(occurrences, path):
    import csv
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["page", "pay_item", "qty", "unit"])
        for o in occurrences:
            w.writerow([o["page"], o["pay_item"], o["qty"], o["unit"]])


def main():
    print("[INFO] Starting QC quantity extraction...")

    if not os.path.exists(PDF_PATH):
        print(f"[ERROR] PDF not found:\n  {PDF_PATH}")
        sys.exit(1)

    occurrences = extract_occurrences(PDF_PATH)
    totals = compute_totals(occurrences)
    overall_expr, per_page_expr = organize_occurrences_expressions(occurrences)

    page_to_sheet = extract_sheet_ids(PDF_PATH)
    per_sheet_expr = build_per_sheet_expr(per_page_expr, page_to_sheet)

    try:
        write_excel_tabulation(
            occurrences=occurrences,
            totals=totals,
            overall_expr=overall_expr,
            per_sheet_expr=per_sheet_expr,
            page_to_sheet=page_to_sheet,
            out_path=OUT_XLSX
        )
        print(f"[OK] Excel written:\n  {OUT_XLSX}")
    except Exception as e:
        print("[ERROR] Failed to write Excel. Writing CSV fallback instead.")
        print("Reason:", repr(e))
        write_csv_fallback(occurrences, OUT_CSV_FALLBACK)
        print(f"[OK] CSV fallback written:\n  {OUT_CSV_FALLBACK}")
        print("\nFull traceback:")
        traceback.print_exc()

    print("[INFO] Done.")


if __name__ == "__main__":
    main()


[INFO] Starting QC quantity extraction...
[INFO] Opened PDF. Pages: 14
[INFO] Processing page 1/14 ...
[INFO] Processing page 6/14 ...
[INFO] Processing page 11/14 ...
[INFO] Processing page 14/14 ...
[INFO] Extracted occurrences: 303
[OK] Excel written:
  C:/Abid/Abid/AI AGENTS/qc PDFS/44038.xlsx
[INFO] Done.


In [2]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------- ----------- 4.7/6.6 MB 22.7 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 22.1 MB/s  0:00:00
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------------------- ----------- 5.0/7.1 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------- 7.1/7.1 MB 22.7 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 23.1 MB/s  0:00:00

   ---------------------------------------- 0/4 [pypdfium2]
   ---------------------------------------- 0/4 [pypdfium2]
  Attempting uninstall: Pillow
   ---------------------------------------- 0/4 [pypdfium2]
    Found existing installation: pillow 12.0.0
   ---------------------------------------- 0/4 [pypdfium2]
   ---------- ----------------------------- 1/4 [Pillow]
    Uninstalling pillow-12.0.0:
   ------

In [1]:
# THIS IS MAIN CODE WITH EXCEL FORMAT CHANGED

import os
import re
import sys
import traceback
from collections import defaultdict

import pdfplumber
from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# ------------------ PATHS (YOUR INPUTS) ------------------
PDF_PATH = r"C:/Abid/Abid/AI AGENTS/qc PDFS/ITS PLANS.pdf"
OUT_XLSX = r"C:/Abid/Abid/AI AGENTS/qc PDFS/pay_item_quantities_539.xlsx"
OUT_CSV_FALLBACK = r"C:/Abid/Abid/AI AGENTS/qc PDFS/pay_item_quantities_fallback.csv"

# ------------------ PATTERNS ------------------
PAY_ITEM_RE = re.compile(r"^\d{3}-\d{1,3}-\d{1,3}$")  # catches 700-141-140
NUM_TOKEN_RE = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d+)?$")

UNITS = {"LF", "EA", "MI", "SF", "AS", "LS", "LU", "DA", "PI", "HR", "LO"}

# For reading sheet numbers at bottom-right
SHEET_ID_RE = re.compile(r"\bIT\s*-\s*\d+\b", re.IGNORECASE)

# ------------------ TUNABLE PARAMETERS ------------------
Y_TOL = 4.0
MAX_HORIZONTAL_DISTANCE = 250
UNIT_GAP_MAX = 70


def autofit_columns(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


def normalize_sheet_id(text):
    return re.sub(r"\s+", "", text.upper())


def extract_sheet_ids(pdf_path):
    """
    Reads bottom-right title block area of each page and extracts sheet IDs like IT-2.
    Returns: dict {page_number: sheet_id}
    """
    page_to_sheet = {}

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            w, h = page.width, page.height

            # Bottom-right crop region
            crop_box = (
                w * 0.78,   # x0
                h * 0.82,   # top
                w * 0.995,  # x1
                h * 0.995   # bottom
            )

            try:
                cropped = page.crop(crop_box)
                text = cropped.extract_text() or ""
            except Exception:
                text = ""

            m = SHEET_ID_RE.search(text)
            if m:
                page_to_sheet[page_num] = normalize_sheet_id(m.group(0))
            else:
                page_to_sheet[page_num] = f"PAGE-{page_num}"

    return page_to_sheet


def extract_occurrences(pdf_path):
    """
    Extract occurrences across ALL pages.
    Each occurrence is a row: {page, pay_item, qty, unit}
    """
    occurrences = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"[INFO] Opened PDF. Pages: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num % 5 == 1 or page_num == total_pages:
                print(f"[INFO] Processing page {page_num}/{total_pages} ...")

            words = page.extract_words(keep_blank_chars=False, use_text_flow=True) or []
            if not words:
                continue

            pay_items, numbers, units = [], [], []

            for w in words:
                text = (w.get("text") or "").strip()
                if not text:
                    continue
                if PAY_ITEM_RE.match(text):
                    pay_items.append(w)
                elif text.upper() in UNITS:
                    units.append(w)
                elif NUM_TOKEN_RE.match(text):
                    numbers.append(w)

            # Build valid (number + unit) pairs: "453" + "LF", "6" + "EA"
            pairs = []
            for q in numbers:
                qy = q["top"]
                for u in units:
                    if abs(u["top"] - qy) <= Y_TOL:
                        gap = u["x0"] - q["x1"]
                        if 0 <= gap <= UNIT_GAP_MAX:
                            pairs.append((q, u))

            # Assign closest pair to each pay item (left OR right), restricted by MAX_HORIZONTAL_DISTANCE
            for pi in pay_items:
                piy = pi["top"]
                pi_center = (pi["x0"] + pi["x1"]) / 2

                candidates = []
                for q, u in pairs:
                    if abs(q["top"] - piy) <= Y_TOL:
                        q_center = (q["x0"] + q["x1"]) / 2
                        dx = abs(q_center - pi_center)
                        if dx <= MAX_HORIZONTAL_DISTANCE:
                            candidates.append((dx, q, u))

                if not candidates:
                    occurrences.append({
                        "page": page_num,
                        "pay_item": pi["text"],
                        "qty": None,
                        "unit": None
                    })
                    continue

                candidates.sort(key=lambda x: x[0])
                _, q, u = candidates[0]

                occurrences.append({
                    "page": page_num,
                    "pay_item": pi["text"],
                    "qty": float(q["text"].replace(",", "")),
                    "unit": u["text"].upper()
                })

    print(f"[INFO] Extracted occurrences: {len(occurrences)}")
    return occurrences


def compute_totals(occurrences):
    totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        totals[(o["pay_item"], o["unit"])] += o["qty"]
    return totals


def organize_occurrences_expressions(occurrences):
    overall_list = defaultdict(list)
    per_page_list = defaultdict(list)

    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        overall_list[(o["pay_item"], o["unit"])].append(o["qty"])
        per_page_list[(o["page"], o["pay_item"], o["unit"])].append(o["qty"])

    def fmt(v):
        return str(int(v)) if float(v).is_integer() else str(v)

    overall_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in overall_list.items()}
    per_page_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in per_page_list.items()}

    return overall_expr, per_page_expr


def sort_sheet_ids(sheet_ids):
    def key_func(s):
        m = re.match(r"([A-Z]+)-(\d+)", s)
        if m:
            return (m.group(1), int(m.group(2)))
        return (s, 999999)
    return sorted(sheet_ids, key=key_func)


def build_per_sheet_expr(per_page_expr, page_to_sheet):
    """
    Convert per-page expressions to per-sheet expressions using sheet IDs.
    Output: {(sheet_id, pay_item, unit): "36+13+273"}
    """
    grouped = defaultdict(list)

    for (page, pay_item, unit), expr in per_page_expr.items():
        sheet_id = page_to_sheet.get(page, f"PAGE-{page}")
        grouped[(sheet_id, pay_item, unit)].append(expr)

    merged = {}
    for key, exprs in grouped.items():
        parts = []
        for e in exprs:
            if e:
                parts.extend(e.split("+"))
        merged[key] = "+".join(parts)

    return merged


def apply_table_border(ws, row_start, row_end, col_start, col_end):
    thin = Side(style="thin", color="000000")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for r in range(row_start, row_end + 1):
        for c in range(col_start, col_end + 1):
            ws.cell(r, c).border = border


def write_excel_tabulation(occurrences, totals, overall_expr, per_sheet_expr, page_to_sheet, out_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Tabulation"

    # ---------------- Styles ----------------
    header_font = Font(name="Arial", size=11, bold=True, italic=True)
    body_font = Font(name="Arial", size=10, italic=True)
    center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center", wrap_text=True)

    # ---------------- Data prep ----------------
    sheet_ids = sort_sheet_ids(sorted(set(page_to_sheet.values())))
    pay_item_keys = sorted(set((o["pay_item"], o["unit"]) for o in occurrences if o["qty"] is not None and o["unit"] is not None))

    # ---------------- Layout ----------------
    # Columns:
    # A = Pay Item No
    # B = Description
    # C = Unit
    # D.. = sheet pairs (PLAN / FINAL)
    # then Grand Total PLAN / FINAL
    start_row = 1

    ws.merge_cells(start_row=1, start_column=4, end_row=1, end_column=3 + len(sheet_ids) * 2)
    ws.cell(1, 4).value = "TABULATION OF QUANTITIES"
    ws.cell(1, 4).font = header_font
    ws.cell(1, 4).alignment = center

    ws.merge_cells(start_row=2, start_column=4, end_row=2, end_column=3 + len(sheet_ids) * 2)
    ws.cell(2, 4).value = "SHEET NUMBERS"
    ws.cell(2, 4).font = header_font
    ws.cell(2, 4).alignment = center

    ws.merge_cells("A3:A4")
    ws["A3"] = "PAY\nITEM\nNO."
    ws["A3"].font = header_font
    ws["A3"].alignment = center

    ws.merge_cells("B3:B4")
    ws["B3"] = "DESCRIPTION"
    ws["B3"].font = header_font
    ws["B3"].alignment = center

    ws.merge_cells("C3:C4")
    ws["C3"] = "UNIT"
    ws["C3"].font = header_font
    ws["C3"].alignment = center

    col = 4
    sheet_plan_col = {}
    for sid in sheet_ids:
        ws.merge_cells(start_row=3, start_column=col, end_row=3, end_column=col + 1)
        ws.cell(3, col).value = sid
        ws.cell(3, col).font = header_font
        ws.cell(3, col).alignment = center

        ws.cell(4, col).value = "PLAN"
        ws.cell(4, col + 1).value = "FINAL"
        ws.cell(4, col).font = header_font
        ws.cell(4, col + 1).font = header_font
        ws.cell(4, col).alignment = center
        ws.cell(4, col + 1).alignment = center

        sheet_plan_col[sid] = col
        col += 2

    grand_plan_col = col
    ws.merge_cells(start_row=3, start_column=grand_plan_col, end_row=3, end_column=grand_plan_col + 1)
    ws.cell(3, grand_plan_col).value = "GRAND\nTOTAL"
    ws.cell(3, grand_plan_col).font = header_font
    ws.cell(3, grand_plan_col).alignment = center

    ws.cell(4, grand_plan_col).value = "PLAN"
    ws.cell(4, grand_plan_col + 1).value = "FINAL"
    ws.cell(4, grand_plan_col).font = header_font
    ws.cell(4, grand_plan_col + 1).font = header_font
    ws.cell(4, grand_plan_col).alignment = center
    ws.cell(4, grand_plan_col + 1).alignment = center

    # ---------------- Fill rows ----------------
    row = 5
    for pay_item, unit in pay_item_keys:
        ws.cell(row, 1).value = pay_item
        ws.cell(row, 2).value = ""   # description left blank as requested
        ws.cell(row, 3).value = unit

        ws.cell(row, 1).font = body_font
        ws.cell(row, 2).font = body_font
        ws.cell(row, 3).font = body_font
        ws.cell(row, 1).alignment = center
        ws.cell(row, 2).alignment = left_align
        ws.cell(row, 3).alignment = center

        # Per-sheet PLAN formulas
        for sid in sheet_ids:
            expr = per_sheet_expr.get((sid, pay_item, unit))
            if expr:
                ws.cell(row, sheet_plan_col[sid]).value = f"={expr}"
            # FINAL stays blank

        # Grand total PLAN formula
        overall = overall_expr.get((pay_item, unit))
        if overall:
            ws.cell(row, grand_plan_col).value = f"={overall}"
        # FINAL stays blank

        row += 1

    # ---------------- Formatting ----------------
    for r in range(5, row):
        for c in range(1, grand_plan_col + 2):
            ws.cell(r, c).font = body_font
            if c == 2:
                ws.cell(r, c).alignment = left_align
            else:
                ws.cell(r, c).alignment = center

    ws.column_dimensions["A"].width = 14
    ws.column_dimensions["B"].width = 55
    ws.column_dimensions["C"].width = 8

    for c in range(4, grand_plan_col + 2):
        ws.column_dimensions[get_column_letter(c)].width = 10

    apply_table_border(ws, 3, row - 1, 1, grand_plan_col + 1)

    # Optional supporting sheets
    ws2 = wb.create_sheet("Occurrences")
    ws2.append(["page", "sheet_id", "pay_item", "qty", "unit"])
    for o in occurrences:
        ws2.append([o["page"], page_to_sheet.get(o["page"], f"PAGE-{o['page']}"), o["pay_item"], o["qty"], o["unit"]])
    autofit_columns(ws2)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wb.save(out_path)


def write_csv_fallback(occurrences, path):
    import csv
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["page", "pay_item", "qty", "unit"])
        for o in occurrences:
            w.writerow([o["page"], o["pay_item"], o["qty"], o["unit"]])


def main():
    print("[INFO] Starting QC quantity extraction...")

    if not os.path.exists(PDF_PATH):
        print(f"[ERROR] PDF not found:\n  {PDF_PATH}")
        sys.exit(1)

    # Detection remains unchanged
    occurrences = extract_occurrences(PDF_PATH)
    totals = compute_totals(occurrences)
    overall_expr, per_page_expr = organize_occurrences_expressions(occurrences)

    # New: get sheet IDs from title block
    page_to_sheet = extract_sheet_ids(PDF_PATH)
    per_sheet_expr = build_per_sheet_expr(per_page_expr, page_to_sheet)

    try:
        write_excel_tabulation(
            occurrences=occurrences,
            totals=totals,
            overall_expr=overall_expr,
            per_sheet_expr=per_sheet_expr,
            page_to_sheet=page_to_sheet,
            out_path=OUT_XLSX
        )
        print(f"[OK] Excel written:\n  {OUT_XLSX}")
    except Exception as e:
        print("[ERROR] Failed to write Excel. Writing CSV fallback instead.")
        print("Reason:", repr(e))
        write_csv_fallback(occurrences, OUT_CSV_FALLBACK)
        print(f"[OK] CSV fallback written:\n  {OUT_CSV_FALLBACK}")
        print("\nFull traceback:")
        traceback.print_exc()

    print("[INFO] Done.")


if __name__ == "__main__":
    main()

[INFO] Starting QC quantity extraction...
[INFO] Opened PDF. Pages: 24
[INFO] Processing page 1/24 ...
[INFO] Processing page 6/24 ...
[INFO] Processing page 11/24 ...
[INFO] Processing page 16/24 ...
[INFO] Processing page 21/24 ...
[INFO] Processing page 24/24 ...
[INFO] Extracted occurrences: 583
[OK] Excel written:
  C:/Abid/Abid/AI AGENTS/qc PDFS/pay_item_quantities_539.xlsx
[INFO] Done.


In [5]:
# THIS IS MAIN CODE (with split-number merge added)

import os
import re
import sys
import traceback
from collections import defaultdict

import pdfplumber
from openpyxl import Workbook
from openpyxl.utils import get_column_letter

# ------------------ PATHS (YOUR INPUTS) ------------------
PDF_PATH = r"C:/Abid/1. I -95 Citrus Blvd/QC tool check/Plans only set 11.pdf"
OUT_XLSX = r"C:/Abid/1. I -95 Citrus Blvd/QC tool check/pay_item_quantities4.xlsx"
OUT_CSV_FALLBACK = r"C:/Abid/1. I -95 Citrus Blvd/QC tool check/pay_item_quantities_fallback.csv"

# ------------------ PATTERNS ------------------
PAY_ITEM_RE = re.compile(r"^\d{3}-\d{1,3}-\d{1,3}$")  # catches 700-141-140
NUM_TOKEN_RE = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d+)?$")

UNITS = {"LF", "EA", "MI", "SF", "AS", "LS", "LU", "DA", "PI", "HR", "LO"}

# ------------------ TUNABLE PARAMETERS ------------------
Y_TOL = 4.0
MAX_HORIZONTAL_DISTANCE = 250
UNIT_GAP_MAX = 70

# New: merge split-digit numbers like "2" + "1" -> "21"
SPLIT_NUM_X_GAP_TOL = 3.0  # typical safe range: 2.0–4.0


def autofit_columns(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


def merge_split_numbers(numbers, y_tol=4.0, x_gap_tol=3.0):
    """
    Merge numeric tokens that are actually one number but got split, e.g. "2" and "1" => "21".
    Only merges tokens on the same row band and with a tiny horizontal gap.
    """
    if not numbers:
        return []

    nums = sorted(numbers, key=lambda w: (w["top"], w["x0"]))
    merged = []
    i = 0

    while i < len(nums):
        cur = dict(nums[i])  # copy
        j = i + 1

        while j < len(nums):
            nxt = nums[j]
            same_row = abs(nxt["top"] - cur["top"]) <= y_tol
            gap = nxt["x0"] - cur["x1"]
            close = 0 <= gap <= x_gap_tol

            # Only merge if the next token is digits and we're not dealing with comma/decimal splits
            if same_row and close and nxt["text"].isdigit() and cur["text"].isdigit():
                cur["text"] = cur["text"] + nxt["text"]
                cur["x1"] = nxt["x1"]
                cur["bottom"] = max(cur["bottom"], nxt["bottom"])
                j += 1
            else:
                break

        merged.append(cur)
        i = j

    return merged


def extract_occurrences(pdf_path):
    """
    Extract occurrences across ALL pages.
    Each occurrence is a row: {page, pay_item, qty, unit}
    """
    occurrences = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"[INFO] Opened PDF. Pages: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num % 5 == 1 or page_num == total_pages:
                print(f"[INFO] Processing page {page_num}/{total_pages} ...")

            words = page.extract_words(keep_blank_chars=False, use_text_flow=True) or []
            if not words:
                continue

            pay_items, numbers, units = [], [], []

            for w in words:
                text = (w.get("text") or "").strip()
                if not text:
                    continue
                if PAY_ITEM_RE.match(text):
                    pay_items.append(w)
                elif text.upper() in UNITS:
                    units.append(w)
                elif NUM_TOKEN_RE.match(text):
                    numbers.append(w)

            # NEW: fix subtle cases like "21" being extracted as "2" and "1"
            numbers = merge_split_numbers(numbers, y_tol=Y_TOL, x_gap_tol=SPLIT_NUM_X_GAP_TOL)

            # Build valid (number + unit) pairs: "453" + "LF", "6" + "EA"
            pairs = []
            for q in numbers:
                qy = q["top"]
                for u in units:
                    if abs(u["top"] - qy) <= Y_TOL:
                        gap = u["x0"] - q["x1"]
                        if 0 <= gap <= UNIT_GAP_MAX:
                            pairs.append((q, u))

            # Assign closest pair to each pay item (left OR right), restricted by MAX_HORIZONTAL_DISTANCE
            for pi in pay_items:
                piy = pi["top"]
                pi_center = (pi["x0"] + pi["x1"]) / 2

                candidates = []
                for q, u in pairs:
                    if abs(q["top"] - piy) <= Y_TOL:
                        q_center = (q["x0"] + q["x1"]) / 2
                        dx = abs(q_center - pi_center)
                        if dx <= MAX_HORIZONTAL_DISTANCE:
                            candidates.append((dx, q, u))

                if not candidates:
                    occurrences.append({
                        "page": page_num,
                        "pay_item": pi["text"],
                        "qty": None,
                        "unit": None
                    })
                    continue

                candidates.sort(key=lambda x: x[0])
                _, q, u = candidates[0]

                occurrences.append({
                    "page": page_num,
                    "pay_item": pi["text"],
                    "qty": float(q["text"].replace(",", "")),
                    "unit": u["text"].upper()
                })

    print(f"[INFO] Extracted occurrences: {len(occurrences)}")
    return occurrences


def compute_totals(occurrences):
    totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        totals[(o["pay_item"], o["unit"])] += o["qty"]
    return totals


def compute_page_totals(occurrences):
    page_totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        page_totals[(o["page"], o["pay_item"], o["unit"])] += o["qty"]
    return page_totals


def organize_occurrences_expressions(occurrences):
    overall_list = defaultdict(list)
    per_page_list = defaultdict(list)

    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        overall_list[(o["pay_item"], o["unit"])].append(o["qty"])
        per_page_list[(o["page"], o["pay_item"], o["unit"])].append(o["qty"])

    def fmt(v):
        return str(int(v)) if float(v).is_integer() else str(v)

    overall_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in overall_list.items()}
    per_page_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in per_page_list.items()}

    return overall_expr, per_page_expr


def write_excel(occurrences, page_totals, totals, overall_expr, per_page_expr, out_path):
    wb = Workbook()

    # Sheet 1: Occurrences
    ws1 = wb.active
    ws1.title = "Occurrences"
    ws1.append(["page", "pay_item", "qty", "unit"])
    for o in occurrences:
        ws1.append([o["page"], o["pay_item"], o["qty"], o["unit"]])
    autofit_columns(ws1)

    # Sheet 2: Page Totals
    ws2 = wb.create_sheet("Page Totals")
    ws2.append(["page", "pay_item", "unit", "total_qty"])
    for (page, item, unit), total in sorted(page_totals.items()):
        ws2.append([page, item, unit, float(total)])
    autofit_columns(ws2)

    # Sheet 3: Grand Totals
    ws3 = wb.create_sheet("Grand Totals")
    ws3.append(["pay_item", "unit", "total_qty"])
    for (item, unit), total in sorted(totals.items()):
        ws3.append([item, unit, float(total)])
    autofit_columns(ws3)

    # Sheet 4: Organized Occurrences (expressions)
    ws4 = wb.create_sheet("Organized Occurrences")
    ws4.append(["scope", "page", "pay_item", "unit", "expression"])

    for (page, item, unit), expr in sorted(per_page_expr.items()):
        ws4.append(["per_page", page, item, unit, expr])

    for (item, unit), expr in sorted(overall_expr.items()):
        ws4.append(["overall", None, item, unit, expr])

    autofit_columns(ws4)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wb.save(out_path)


def write_csv_fallback(occurrences, path):
    import csv
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["page", "pay_item", "qty", "unit"])
        for o in occurrences:
            w.writerow([o["page"], o["pay_item"], o["qty"], o["unit"]])


def main():
    print("[INFO] Starting QC quantity extraction...")

    if not os.path.exists(PDF_PATH):
        print(f"[ERROR] PDF not found:\n  {PDF_PATH}")
        sys.exit(1)

    occurrences = extract_occurrences(PDF_PATH)

    totals = compute_totals(occurrences)
    page_totals = compute_page_totals(occurrences)
    overall_expr, per_page_expr = organize_occurrences_expressions(occurrences)

    try:
        write_excel(occurrences, page_totals, totals, overall_expr, per_page_expr, OUT_XLSX)
        print(f"[OK] Excel written:\n  {OUT_XLSX}")
    except Exception as e:
        print("[ERROR] Failed to write Excel. Writing CSV fallback instead.")
        print("Reason:", repr(e))
        write_csv_fallback(occurrences, OUT_CSV_FALLBACK)
        print(f"[OK] CSV fallback written:\n  {OUT_CSV_FALLBACK}")
        print("\nFull traceback:")
        traceback.print_exc()

    print("[INFO] Done.")


if __name__ == "__main__":
    main()

[INFO] Starting QC quantity extraction...
[INFO] Opened PDF. Pages: 1
[INFO] Processing page 1/1 ...
[INFO] Extracted occurrences: 34
[OK] Excel written:
  C:/Abid/1. I -95 Citrus Blvd/QC tool check/pay_item_quantities4.xlsx
[INFO] Done.


In [3]:
import os
import re
import sys
import traceback
from collections import defaultdict

import pdfplumber
from openpyxl import Workbook
from openpyxl.utils import get_column_letter

# ------------------ PATHS (YOUR INPUTS) ------------------
PDF_PATH = r"C:/Abid/1. I -95 Citrus Blvd/QC tool check/QC Check 3.pdf"
OUT_XLSX = r"C:/Abid/1. I -95 Citrus Blvd/QC tool check/pay_item_quantities3.xlsx"
OUT_CSV_FALLBACK = r"C:/Abid/1. I -95 Citrus Blvd/QC tool check/pay_item_quantities_fallback.csv"

# ------------------ PATTERNS ------------------
PAY_ITEM_RE = re.compile(r"^\d{3}-\d{1,3}-\d{1,3}$")  # catches 700-141-140
NUM_TOKEN_RE = re.compile(r"^\d{1,3}(?:,\d{3})*(?:\.\d+)?$")

# Units: expanded (as you asked)
UNITS = {"LF", "EA", "MI", "SF", "AS", "LS", "LU", "DA", "PI", "HR", "LO"}

# Quantity+unit regex used in LOCAL window (supports "137 LF" and "137LF")
QTY_UNIT_RE = re.compile(
    r"\b(\d{1,3}(?:,\d{3})*(?:\.\d+)?)\s*(" + "|".join(sorted(UNITS, key=len, reverse=True)) + r")\b",
    re.IGNORECASE
)

PAREN_RE = re.compile(r"\([^)]*\)")  # remove ( ... ) so we ignore "(2 RUNS @ 100 LF)"


# ------------------ TUNABLE PARAMETERS ------------------
Y_TOL = 6.0
MAX_HORIZONTAL_DISTANCE = 350  # local window width to the right of pay item (and a bit to the left)
UNIT_GAP_MAX = 70              # kept for reference; not used in the local-regex method


def autofit_columns(ws):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


def extract_occurrences(pdf_path):
    """
    Extract occurrences across ALL pages.
    Uses a LOCAL WINDOW around each pay item row to find the nearest qty+unit text.
    This prevents wrong matches like picking '1 LF' or '5 LF' from unrelated nearby numbers.
    """
    occurrences = []

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        print(f"[INFO] Opened PDF. Pages: {total_pages}")

        for page_num, page in enumerate(pdf.pages, start=1):
            if page_num % 5 == 1 or page_num == total_pages:
                print(f"[INFO] Processing page {page_num}/{total_pages} ...")

            words = page.extract_words(keep_blank_chars=False, use_text_flow=True) or []
            if not words:
                continue

            # Collect pay item tokens
            pay_items = []
            for w in words:
                text = (w.get("text") or "").strip()
                if text and PAY_ITEM_RE.match(text):
                    pay_items.append(w)

            # For each pay item, crop a local region and regex-search for qty+unit
            for pi in pay_items:
                # Local search window around the pay item row:
                # - small vertical band (top/bottom +- Y_TOL)
                # - horizontal window includes both left and right nearby qty columns
                x0 = max(0, pi["x0"] - 120)  # allow qty on left
                x1 = min(page.width, pi["x1"] + MAX_HORIZONTAL_DISTANCE)  # allow qty on right
                y0 = max(0, pi["top"] - Y_TOL)
                y1 = min(page.height, pi["bottom"] + Y_TOL)

                local = page.crop((x0, y0, x1, y1))

                # Extract local text and clean it
                local_text = (local.extract_text() or "").replace("\n", " ")
                local_text = PAREN_RE.sub(" ", local_text)  # remove parenthetical calculations

                matches = list(QTY_UNIT_RE.finditer(local_text))

                if not matches:
                    occurrences.append({
                        "page": page_num,
                        "pay_item": pi["text"],
                        "qty": None,
                        "unit": None
                    })
                    continue

                # Choose the FIRST qty+unit found in the local band.
                # This typically grabs the main quantity (e.g., "137 LF") and ignores later notes.
                m = matches[0]
                qty = float(m.group(1).replace(",", ""))
                unit = m.group(2).upper()

                occurrences.append({
                    "page": page_num,
                    "pay_item": pi["text"],
                    "qty": qty,
                    "unit": unit
                })

    print(f"[INFO] Extracted occurrences: {len(occurrences)}")
    return occurrences


def compute_totals(occurrences):
    totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        totals[(o["pay_item"], o["unit"])] += o["qty"]
    return totals


def compute_page_totals(occurrences):
    page_totals = defaultdict(float)
    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        page_totals[(o["page"], o["pay_item"], o["unit"])] += o["qty"]
    return page_totals


def organize_occurrences_expressions(occurrences):
    overall_list = defaultdict(list)
    per_page_list = defaultdict(list)

    for o in occurrences:
        if o["qty"] is None or o["unit"] is None:
            continue
        overall_list[(o["pay_item"], o["unit"])].append(o["qty"])
        per_page_list[(o["page"], o["pay_item"], o["unit"])].append(o["qty"])

    def fmt(v):
        return str(int(v)) if float(v).is_integer() else str(v)

    overall_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in overall_list.items()}
    per_page_expr = {k: "+".join(fmt(v) for v in vals) for k, vals in per_page_list.items()}

    return overall_expr, per_page_expr


def write_excel(occurrences, page_totals, totals, overall_expr, per_page_expr, out_path):
    wb = Workbook()

    # Sheet 1: Occurrences
    ws1 = wb.active
    ws1.title = "Occurrences"
    ws1.append(["page", "pay_item", "qty", "unit"])
    for o in occurrences:
        ws1.append([o["page"], o["pay_item"], o["qty"], o["unit"]])
    autofit_columns(ws1)

    # Sheet 2: Page Totals
    ws2 = wb.create_sheet("Page Totals")
    ws2.append(["page", "pay_item", "unit", "total_qty"])
    for (page, item, unit), total in sorted(page_totals.items()):
        ws2.append([page, item, unit, float(total)])
    autofit_columns(ws2)

    # Sheet 3: Grand Totals
    ws3 = wb.create_sheet("Grand Totals")
    ws3.append(["pay_item", "unit", "total_qty"])
    for (item, unit), total in sorted(totals.items()):
        ws3.append([item, unit, float(total)])
    autofit_columns(ws3)

    # Sheet 4: Organized Occurrences (expressions)
    ws4 = wb.create_sheet("Organized Occurrences")
    ws4.append(["scope", "page", "pay_item", "unit", "expression"])

    for (page, item, unit), expr in sorted(per_page_expr.items()):
        ws4.append(["per_page", page, item, unit, expr])

    for (item, unit), expr in sorted(overall_expr.items()):
        ws4.append(["overall", None, item, unit, expr])

    autofit_columns(ws4)

    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    wb.save(out_path)


def write_csv_fallback(occurrences, path):
    import csv
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["page", "pay_item", "qty", "unit"])
        for o in occurrences:
            w.writerow([o["page"], o["pay_item"], o["qty"], o["unit"]])


def main():
    print("[INFO] Starting QC quantity extraction...")

    if not os.path.exists(PDF_PATH):
        print(f"[ERROR] PDF not found:\n  {PDF_PATH}")
        sys.exit(1)

    occurrences = extract_occurrences(PDF_PATH)

    totals = compute_totals(occurrences)
    page_totals = compute_page_totals(occurrences)
    overall_expr, per_page_expr = organize_occurrences_expressions(occurrences)

    try:
        write_excel(occurrences, page_totals, totals, overall_expr, per_page_expr, OUT_XLSX)
        print(f"[OK] Excel written:\n  {OUT_XLSX}")
    except Exception as e:
        print("[ERROR] Failed to write Excel. Writing CSV fallback instead.")
        print("Reason:", repr(e))
        write_csv_fallback(occurrences, OUT_CSV_FALLBACK)
        print(f"[OK] CSV fallback written:\n  {OUT_CSV_FALLBACK}")
        print("\nFull traceback:")
        traceback.print_exc()

    print("[INFO] Done.")


if __name__ == "__main__":
    main()


[INFO] Starting QC quantity extraction...
[INFO] Opened PDF. Pages: 17
[INFO] Processing page 1/17 ...
[INFO] Processing page 6/17 ...
[INFO] Processing page 11/17 ...
[INFO] Processing page 16/17 ...
[INFO] Processing page 17/17 ...
[INFO] Extracted occurrences: 449
[OK] Excel written:
  C:/Abid/1. I -95 Citrus Blvd/QC tool check/pay_item_quantities3.xlsx
[INFO] Done.
